# 📊 Customer Churn Analysis & Prediction
**Author:** Nagaswetha Kaikala
**Tools:** Python · SQL · Pandas · Scikit-learn · Matplotlib · Seaborn
**Dataset:** 3 related tables — 5,000 customers, 177K transactions, 8,887 support tickets

---
### Project Overview
End-to-end data project covering:
- **Data Engineering:** Multi-table data integration & SQL-style analysis
- **Data Analysis:** EDA across customer, transaction & support data
- **Machine Learning:** Churn prediction model (Logistic Regression + Random Forest)
- **BI:** KPI dashboard-style visualizations for business stakeholders
- **Outcome:** Identify at-risk customers & recommend retention strategies


## 1. Setup & Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, accuracy_score)
from sklearn.pipeline import Pipeline
import warnings, os
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
Plt_COLORS = ['#2c3e7a','#1a6b3c','#e67e22','#c0392b','#8e44ad']
os.makedirs('charts', exist_ok=True)

print('✅ All libraries loaded!')
print(f'   Pandas: {pd.__version__} | Sklearn: imported')

## 2. Load & Integrate 3 Datasets (Data Engineering Step)

In [ ]:
# Load all 3 tables
customers  = pd.read_csv('customers.csv')
transactions = pd.read_csv('transactions.csv')
tickets    = pd.read_csv('support_tickets.csv')

print('=== TABLE SHAPES ===')
print(f'customers:     {customers.shape[0]:,} rows × {customers.shape[1]} cols')
print(f'transactions:  {transactions.shape[0]:,} rows × {transactions.shape[1]} cols')
print(f'support_tickets: {tickets.shape[0]:,} rows × {tickets.shape[1]} cols')

print('\n=== CUSTOMERS TABLE PREVIEW ===')
customers.head()

In [ ]:
# SQL-style aggregations — join transactions & tickets to customers
txn_agg = transactions.groupby('customer_id').agg(
    total_transactions = ('transaction_id','count'),
    total_revenue      = ('amount','sum'),
    avg_txn_amount     = ('amount','mean'),
    late_payments      = ('late_payment','sum'),
    last_txn_date      = ('txn_date','max')
).reset_index()
txn_agg['late_payment_rate'] = (txn_agg['late_payments'] / txn_agg['total_transactions'] * 100).round(2)

ticket_agg = tickets.groupby('customer_id').agg(
    total_tickets      = ('ticket_id','count'),
    avg_resolve_days   = ('days_to_resolve','mean'),
    escalated_count    = ('resolution', lambda x: (x=='Escalated').sum()),
    avg_post_satisfaction = ('satisfaction_after','mean')
).reset_index()

# Master table — like a SQL JOIN
df = customers.merge(txn_agg, on='customer_id', how='left')
df = df.merge(ticket_agg, on='customer_id', how='left')
df = df.fillna(0)

print(f'✅ Master table created: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   Churn rate: {df["churned"].mean()*100:.1f}%')
print(f'   Total revenue: ${df["total_revenue"].sum():,.0f}')
df.head()

## 3. Data Quality Check

In [ ]:
print('=== MISSING VALUES ===')
print(df.isnull().sum()[df.isnull().sum()>0] if df.isnull().sum().sum()>0 else 'No missing values!')

print('\n=== DATA TYPES ===')
print(df.dtypes)

print('\n=== KEY STATISTICS ===')
df[['tenure_months','monthly_charges','total_charges','satisfaction_score',
    'support_calls','total_transactions','late_payment_rate']].describe().round(2)

## 4. Exploratory Data Analysis (EDA)

### 4A. Churn Overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Churn distribution
churn_counts = df['churned'].value_counts()
axes[0].pie(churn_counts, labels=['Retained','Churned'], autopct='%1.1f%%',
            colors=['#1a6b3c','#c0392b'], startangle=90, wedgeprops=dict(width=0.6))
axes[0].set_title('Overall Churn Rate', fontweight='bold', fontsize=12)

# Churn by Plan
plan_churn = df.groupby('plan')['churned'].mean() * 100
axes[1].bar(plan_churn.index, plan_churn.values, color=['#2c3e7a','#1a6b3c','#e67e22'])
for i,(idx,val) in enumerate(plan_churn.items()):
    axes[1].text(i, val+0.3, f'{val:.1f}%', ha='center', fontweight='bold')
axes[1].set_title('Churn Rate by Plan', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_ylim(0, plan_churn.max()*1.25)

# Churn by Segment
seg_churn = df.groupby('segment')['churned'].mean() * 100
axes[2].barh(seg_churn.index, seg_churn.values, color='#2c3e7a')
axes[2].set_title('Churn Rate by Segment', fontweight='bold', fontsize=12)
axes[2].set_xlabel('Churn Rate (%)')

plt.suptitle('Churn Overview', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('charts/01_churn_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Churn rate by plan:')
print(plan_churn.round(1).to_string())

### 4B. Tenure & Charges Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Tenure distribution by churn
df[df['churned']==0]['tenure_months'].hist(ax=axes[0], bins=30, alpha=0.7,
    color='#1a6b3c', label='Retained', density=True)
df[df['churned']==1]['tenure_months'].hist(ax=axes[0], bins=30, alpha=0.7,
    color='#c0392b', label='Churned', density=True)
axes[0].set_title('Tenure Distribution: Churned vs Retained', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Tenure (months)')
axes[0].set_ylabel('Density')
axes[0].legend()
axes[0].axvline(df[df['churned']==1]['tenure_months'].median(), color='#c0392b',
                linestyle='--', linewidth=1.5, label=f'Churn median')

# Monthly charges boxplot
churn_labels = {0:'Retained', 1:'Churned'}
df['churn_label'] = df['churned'].map(churn_labels)
df.boxplot(column='monthly_charges', by='churn_label', ax=axes[1],
           boxprops=dict(color='#2c3e7a'),
           medianprops=dict(color='#c0392b', linewidth=2))
axes[1].set_title('Monthly Charges Distribution', fontweight='bold', fontsize=12)
axes[1].set_xlabel('')
axes[1].set_ylabel('Monthly Charges ($)')
plt.suptitle('')

plt.tight_layout()
plt.savefig('charts/02_tenure_charges.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Median tenure — Churned: {df[df["churned"]==1]["tenure_months"].median():.0f} months')
print(f'Median tenure — Retained: {df[df["churned"]==0]["tenure_months"].median():.0f} months')

### 4C. Support Calls & Satisfaction

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Support calls vs churn
support_churn = df.groupby('support_calls')['churned'].mean() * 100
support_churn = support_churn[support_churn.index <= 8]
axes[0].bar(support_churn.index, support_churn.values, color=['#1a6b3c' if v<30 else '#c0392b' for v in support_churn.values])
axes[0].set_title('Churn Rate by # of Support Calls', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Number of Support Calls')
axes[0].set_ylabel('Churn Rate (%)')
for i, (idx, val) in enumerate(support_churn.items()):
    axes[0].text(idx, val+0.5, f'{val:.0f}%', ha='center', fontsize=8, fontweight='bold')

# Satisfaction score distribution
df[df['churned']==0]['satisfaction_score'].hist(ax=axes[1], bins=20, alpha=0.7,
    color='#1a6b3c', label='Retained', density=True)
df[df['churned']==1]['satisfaction_score'].hist(ax=axes[1], bins=20, alpha=0.7,
    color='#c0392b', label='Churned', density=True)
axes[1].set_title('Satisfaction Score: Churned vs Retained', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Satisfaction Score (1-10)')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.suptitle('Support & Satisfaction Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('charts/03_support_satisfaction.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Avg satisfaction — Churned:  {df[df["churned"]==1]["satisfaction_score"].mean():.2f}')
print(f'Avg satisfaction — Retained: {df[df["churned"]==0]["satisfaction_score"].mean():.2f}')

### 4D. Revenue & Late Payment Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Revenue by plan
rev_plan = df.groupby('plan')['total_revenue'].sum()
axes[0].bar(rev_plan.index, rev_plan.values, color=['#2c3e7a','#1a6b3c','#e67e22'])
for i,(idx,val) in enumerate(rev_plan.items()):
    axes[0].text(i, val+5000, f'${val/1e6:.2f}M', ha='center', fontweight='bold', fontsize=10)
axes[0].set_title('Total Revenue by Plan', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Revenue ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M'))

# Late payment rate by churn
late_data = df.groupby('churned')['late_payment_rate'].mean()
bar_cols = ['#1a6b3c','#c0392b']
bars = axes[1].bar(['Retained','Churned'], late_data.values, color=bar_cols, width=0.4)
for bar,val in zip(bars, late_data.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                 f'{val:.1f}%', ha='center', fontweight='bold', fontsize=11)
axes[1].set_title('Avg Late Payment Rate: Churned vs Retained', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Late Payment Rate (%)')
axes[1].set_ylim(0, late_data.max()*1.3)

plt.suptitle('Revenue & Payment Behaviour', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('charts/04_revenue_payments.png', dpi=150, bbox_inches='tight')
plt.show()

### 4E. Correlation Heatmap

In [ ]:
num_cols = ['tenure_months','monthly_charges','total_charges','support_calls',
            'satisfaction_score','total_transactions','late_payment_rate',
            'avg_resolve_days','escalated_count','churned']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn_r', ax=ax,
            square=True, linewidths=0.5, vmin=-1, vmax=1,
            annot_kws={'size':8})
ax.set_title('Feature Correlation Heatmap\n(Focus: correlation with "churned" column)',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('charts/05_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top correlations with churned:')
print(corr['churned'].drop('churned').sort_values().to_string())

## 5. SQL-Style Business Analysis

In [ ]:
# SQL Query 1: Revenue at risk from churned customers
at_risk_revenue = df[df['churned']==1]['monthly_charges'].sum()
total_revenue   = df['monthly_charges'].sum()
print('=== BUSINESS INTELLIGENCE METRICS ===')
print(f'Total monthly revenue:          ${total_revenue:,.0f}')
print(f'Revenue at risk (churned):      ${at_risk_revenue:,.0f} ({at_risk_revenue/total_revenue*100:.1f}% of total)')

# SQL Query 2: Customer Lifetime Value by segment
print('\n=== CUSTOMER LIFETIME VALUE BY SEGMENT ===')
clv = df.groupby('segment').agg(
    customers=('customer_id','count'),
    avg_tenure=('tenure_months','mean'),
    avg_monthly=('monthly_charges','mean'),
    churn_rate=('churned','mean'),
    avg_total_revenue=('total_revenue','mean')
).round(2)
clv['churn_rate'] = (clv['churn_rate']*100).round(1).astype(str)+'%'
print(clv.to_string())

# SQL Query 3: High-risk customer profile
print('\n=== HIGH-RISK CUSTOMER PROFILE ===')
high_risk = df[(df['churned']==0) &  # still active
               (df['satisfaction_score'] < 5) &
               (df['support_calls'] >= 3) &
               (df['tenure_months'] < 12)]
print(f'Active customers at HIGH churn risk: {len(high_risk):,}')
print(f'Monthly revenue at risk:             ${high_risk["monthly_charges"].sum():,.0f}')
print(f'Avg satisfaction of this group:      {high_risk["satisfaction_score"].mean():.2f}')

## 6. Machine Learning — Churn Prediction Model

### 6A. Feature Engineering & Preprocessing

In [ ]:
# Select features
feature_cols = [
    'gender','age','senior_citizen','tenure_months','plan',
    'monthly_charges','total_charges','payment_method',
    'paperless_billing','has_addon_service','dependents',
    'support_calls','satisfaction_score','total_transactions',
    'late_payment_rate','avg_resolve_days','escalated_count'
]

X = df[feature_cols].copy()
y = df['churned']

# Encode categorical columns
le = LabelEncoder()
for col in ['gender','plan','payment_method']:
    X[col] = le.fit_transform(X[col])

print(f'Feature matrix shape: {X.shape}')
print(f'Target distribution:\n{y.value_counts(normalize=True).round(3)*100}')
print(f'\nFeatures used: {list(X.columns)}')

### 6B. Train-Test Split & Model Training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Training set: {len(X_train):,} samples')
print(f'Test set:     {len(X_test):,} samples')

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Model 1: Logistic Regression
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)
lr_prob = lr.predict_proba(X_test_scaled)[:,1]

# Model 2: Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:,1]

# Model 3: Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=100, random_state=42, learning_rate=0.1)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
gb_prob = gb.predict_proba(X_test)[:,1]

print('\n=== MODEL PERFORMANCE SUMMARY ===')
for name, pred, prob in [('Logistic Regression', lr_pred, lr_prob),
                          ('Random Forest',       rf_pred, rf_prob),
                          ('Gradient Boosting',   gb_pred, gb_prob)]:
    acc = accuracy_score(y_test, pred)
    auc = roc_auc_score(y_test, prob)
    print(f'{name:<22} Accuracy: {acc:.3f}  |  AUC-ROC: {auc:.3f}')

### 6C. Model Evaluation — ROC Curve & Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC Curves
for name, prob, color in [('Logistic Regression', lr_prob, '#2c3e7a'),
                            ('Random Forest', rf_prob, '#1a6b3c'),
                            ('Gradient Boosting', gb_prob, '#e67e22')]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    axes[0].plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc:.3f})')
axes[0].plot([0,1],[0,1],'k--', linewidth=1, label='Random Classifier')
axes[0].set_title('ROC Curve Comparison — All 3 Models', fontweight='bold', fontsize=12)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(fontsize=9)
axes[0].fill_between([0,1],[0,1], alpha=0.05, color='gray')

# Confusion Matrix (best model — Random Forest)
cm = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Retained','Churned'],
            yticklabels=['Retained','Churned'])
axes[1].set_title('Confusion Matrix — Random Forest', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.suptitle('Model Evaluation', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('charts/06_model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== CLASSIFICATION REPORT (Random Forest) ===')
print(classification_report(y_test, rf_pred, target_names=['Retained','Churned']))

### 6D. Feature Importance

In [ ]:
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors_fi = ['#c0392b' if v >= feat_imp.quantile(0.75) else '#2c3e7a' for v in feat_imp.values]
ax.barh(feat_imp.index, feat_imp.values, color=colors_fi)
ax.set_title('Feature Importance — Random Forest Churn Model\n(🔴 Top predictors of churn)',
             fontweight='bold', fontsize=12)
ax.set_xlabel('Importance Score')
for i, (idx, val) in enumerate(feat_imp.items()):
    ax.text(val+0.001, i, f'{val:.3f}', va='center', fontsize=7.5)

plt.tight_layout()
plt.savefig('charts/07_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 churn predictors:')
print(feat_imp.tail(5).round(4).to_string())

## 7. Retention Strategy — Identify At-Risk Customers

In [ ]:
# Predict churn probability for ALL active customers
active = df[df['churned']==0].copy()
X_active = active[feature_cols].copy()
for col in ['gender','plan','payment_method']:
    X_active[col] = le.fit_transform(X_active[col])

active['churn_probability'] = rf.predict_proba(X_active)[:,1]
active['risk_tier'] = pd.cut(active['churn_probability'],
    bins=[0,0.3,0.5,0.7,1.0],
    labels=['Low Risk','Medium Risk','High Risk','Critical Risk'])

print('=== AT-RISK CUSTOMER TIERS ===')
risk_summary = active.groupby('risk_tier', observed=True).agg(
    customers=('customer_id','count'),
    monthly_revenue_at_risk=('monthly_charges','sum'),
    avg_churn_prob=('churn_probability','mean')
).round(2)
risk_summary['avg_churn_prob'] = (risk_summary['avg_churn_prob']*100).round(1).astype(str)+'%'
print(risk_summary.to_string())

print('\n=== TOP 10 HIGHEST RISK ACTIVE CUSTOMERS ===')
top_risk = active.nlargest(10,'churn_probability')[['customer_id','plan','tenure_months',
    'monthly_charges','satisfaction_score','support_calls','churn_probability']]
top_risk['churn_probability'] = (top_risk['churn_probability']*100).round(1).astype(str)+'%'
print(top_risk.to_string(index=False))

## 8. Key Findings & Business Recommendations

---

### 📍 Finding 1: New Customers Churn the Most
- Customers with **tenure < 12 months** have 2–3x higher churn rate than long-term customers
- **Action:** Implement a 90-day onboarding program with proactive check-ins for new customers

---

### 📍 Finding 2: Basic Plan is Leaking Revenue
- Basic plan has the highest churn rate (~35%+)
- **Action:** Offer Basic plan customers a discounted Standard upgrade in month 3–6

---

### 🚨 Finding 3: Support Calls = Churn Signal (Critical)
- Churn rate jumps significantly with each additional support call
- Customers with 4+ support calls have near-critical churn probability
- **Action:** Flag customers after 2nd support call for proactive retention outreach

---

### 📍 Finding 4: Satisfaction Score is the Strongest Predictor
- Churned customers avg satisfaction: ~5.0 vs. Retained: ~8.0
- **Action:** Send automated satisfaction surveys monthly; escalate scores below 6

---

### 📍 Finding 5: Late Payments Predict Churn
- Churned customers have 2–3x higher late payment rates
- **Action:** Trigger a retention offer when a customer misses 2+ consecutive payments

---

### ✅ Recommendations Summary
| Priority | Action | Expected Impact |
|---|---|---|
| 🔴 Critical | Proactive outreach after 2nd support call | Reduce churn by 15–20% |
| 🔴 Critical | Monthly satisfaction survey + escalation | Early warning system |
| 🟠 High | 90-day onboarding program for new customers | Cut early churn in half |
| 🟠 High | Upgrade offer for Basic plan at month 3–6 | Reduce Basic plan churn |
| 🟡 Medium | Payment reminder + retention offer after 2 late payments | Reduce late-payment churn |


In [ ]:
print('='*55)
print('  CUSTOMER CHURN ANALYSIS — FINAL SUMMARY')
print('='*55)
print(f'  Total Customers:        {len(df):,}')
print(f'  Churned Customers:      {df["churned"].sum():,} ({df["churned"].mean()*100:.1f}%)')
print(f'  Total Revenue:          ${df["total_revenue"].sum():,.0f}')
print(f'  Revenue at Risk:        ${df[df["churned"]==1]["monthly_charges"].sum():,.0f}/month')
print(f'  Best ML Model:          Random Forest')
print(f'  Model AUC-ROC:          {roc_auc_score(y_test, rf_prob):.3f}')
print(f'  Model Accuracy:         {accuracy_score(y_test, rf_pred)*100:.1f}%')
print(f'  Critical Risk Customers:{len(active[active["risk_tier"]=="Critical Risk"]):,} active customers')
print('='*55)
print('  Charts saved in ./charts/')
print('  Analysis by: Nagaswetha Kaikala')
print('='*55)